<a href="https://colab.research.google.com/github/ravneetkaur1029-star/real-estate-analytics-pipeline/blob/main/notebooks/pyspark_payment_analytics_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



```
# Payment Analytics Pipeline (PySpark)

## Business Problem
Analyze tenant payment behavior to:
- Identify late payment trends
- Detect high-risk tenants
- Calculate revenue and late fees
- Support financial decision-making

```



In [ ]:
!wget https://raw.githubusercontent.com/ravneetkaur1029-star/csv-file-for-pyspark/refs/heads/main/payments.csv
!wget https://raw.githubusercontent.com/ravneetkaur1029-star/csv-file-for-pyspark/refs/heads/main/Tenants.csv

--2026-05-05 20:45:29--  https://raw.githubusercontent.com/ravneetkaur1029-star/csv-file-for-pyspark/refs/heads/main/payments.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 607 [text/plain]
Saving to: ‘payments.csv.4’

payments.csv.4      100%[===================>]     607  --.-KB/s    in 0s      

2026-05-05 20:45:29 (29.6 MB/s) - ‘payments.csv.4’ saved [607/607]

--2026-05-05 20:45:29--  https://raw.githubusercontent.com/ravneetkaur1029-star/csv-file-for-pyspark/refs/heads/main/Tenants.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 2

In [ ]:
!pip install pyspark

from pyspark.sql import SparkSession
import pandas as pd

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("RealEstate") \
    .getOrCreate()

print("Spark started!")

Spark started!


In [2]:
from pyspark.sql import SparkSession
import pandas as pd
from io import StringIO

# Start Spark
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("RealEstatePipeline") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("="*50)
print("CREATING ALL DATASETS")
print("="*50)

# ============================================================
# PROPERTIES
# ============================================================
properties_data = """property_id,property_name,city,state,total_units,monthly_rent,property_type,year_built
1,Sunset Apartments,Houston,TX,20,2000,Apartment,2005
2,River View,Houston,TX,15,2200,Apartment,2010
3,Downtown Lofts,Dallas,TX,25,2500,Loft,2015
4,Lake Shore,Dallas,TX,30,1800,Apartment,2000
5,Ocean Breeze,Miami,FL,22,2300,Condo,2018
6,Highland Park,Austin,TX,18,2100,Apartment,2012
7,Metro Heights,Houston,TX,28,1900,Apartment,2008
8,Riverside Plaza,Dallas,TX,24,2400,Condo,2016
9,Greenwood Estates,Miami,FL,20,2600,Townhouse,2019
10,Bayou Village,Houston,TX,16,2150,Apartment,2003"""

# ============================================================
# TENANTS
# ============================================================
tenants_data = """tenant_id,tenant_name,email,phone,move_in_date,employment_status,credit_score
1,John Smith,john@email.com,555-0101,2023-01-01,Employed,720
2,Emma Davis,emma@email.com,555-0102,2023-03-01,Employed,680
3,Robert Brown,robert@email.com,555-0103,2023-06-01,Employed,750
4,Maria Garcia,maria@email.com,555-0104,2023-09-01,Self Employed,700
5,David Wilson,david@email.com,555-0105,2023-01-01,Employed,730
6,Sarah Lee,sarah@email.com,555-0106,2023-04-01,Employed,690
7,James Taylor,james@email.com,555-0107,2023-07-01,Employed,760
8,Linda Martinez,linda@email.com,555-0108,2023-01-01,Unemployed,620
9,Michael Anderson,michael@email.com,555-0109,2023-05-01,Employed,740
10,Jennifer Thomas,jennifer@email.com,555-0110,2023-08-01,Employed,710
11,William Jackson,william@email.com,555-0111,2023-02-01,Employed,725
12,Patricia White,patricia@email.com,555-0112,2023-06-01,Self Employed,695
13,Charles Harris,charles@email.com,555-0113,2023-03-01,Employed,755
14,Barbara Clark,barbara@email.com,555-0114,2023-07-01,Employed,715
15,Richard Lewis,richard@email.com,555-0115,2023-09-01,Employed,735"""

# ============================================================
# LEASES
# ============================================================
leases_data = """lease_id,tenant_id,property_id,unit_number,monthly_rent,original_rent,lease_start,lease_end,status,lease_type,renewed
1,1,1,101,2200,2000,2023-01-01,2024-12-31,active,Annual,Yes
2,2,1,102,2500,2200,2023-03-01,2024-06-30,active,Annual,No
3,3,2,201,1800,1800,2023-06-01,2024-09-30,active,Annual,Yes
4,4,2,202,2100,1900,2023-09-01,2024-12-31,active,Annual,No
5,5,3,301,2500,2300,2023-01-01,2024-03-31,active,Annual,Yes
6,6,3,302,2200,2000,2023-04-01,2024-08-31,active,Annual,No
7,7,4,401,1900,1900,2023-07-01,2024-12-31,active,Annual,Yes
8,8,4,402,2000,1800,2023-01-01,2024-06-30,active,Monthly,No
9,9,5,501,2300,2200,2023-05-01,2024-10-31,active,Annual,Yes
10,10,5,502,2400,2100,2023-08-01,2024-12-31,active,Annual,No
11,11,6,601,2100,2000,2023-02-01,2024-12-31,active,Annual,Yes
12,12,6,602,2150,2000,2023-06-01,2024-09-30,active,Annual,No
13,13,7,701,1900,1800,2023-03-01,2024-12-31,active,Annual,Yes
14,14,7,702,1950,1900,2023-07-01,2024-06-30,active,Annual,No
15,15,8,801,2400,2300,2023-09-01,2024-12-31,active,Annual,Yes"""

# ============================================================
# PAYMENTS
# ============================================================
payments_data = """payment_id,tenant_id,lease_id,amount,payment_date,due_date,days_late,payment_method,payment_status
1,1,1,2200,2024-01-03,2024-01-01,2,Bank Transfer,Paid
2,1,1,2200,2024-02-06,2024-02-01,5,Bank Transfer,Paid
3,1,1,2200,2024-03-09,2024-03-01,8,Bank Transfer,Paid
4,2,2,2500,2024-01-10,2024-01-01,9,Check,Paid
5,2,2,2500,2024-02-12,2024-02-01,11,Check,Paid
6,2,2,2500,2024-03-08,2024-03-01,7,Check,Paid
7,3,3,1800,2024-01-01,2024-01-01,0,Bank Transfer,Paid
8,3,3,1800,2024-02-01,2024-02-01,0,Bank Transfer,Paid
9,3,3,1800,2024-03-01,2024-03-01,0,Bank Transfer,Paid
10,4,4,2100,2024-01-05,2024-01-01,4,Credit Card,Paid
11,4,4,2100,2024-02-07,2024-02-01,6,Credit Card,Paid
12,4,4,2100,2024-03-01,2024-03-01,0,Credit Card,Paid
13,5,5,2500,2024-01-01,2024-01-01,0,Bank Transfer,Paid
14,5,5,2500,2024-02-01,2024-02-01,0,Bank Transfer,Paid
15,5,5,2500,2024-03-15,2024-03-01,14,Bank Transfer,Late
16,6,6,2200,2024-01-08,2024-01-01,7,Check,Paid
17,6,6,2200,2024-02-10,2024-02-01,9,Check,Paid
18,6,6,2200,2024-03-12,2024-03-01,11,Check,Late
19,7,7,1900,2024-01-01,2024-01-01,0,Bank Transfer,Paid
20,7,7,1900,2024-02-01,2024-02-01,0,Bank Transfer,Paid
21,7,7,1900,2024-03-01,2024-03-01,0,Bank Transfer,Paid
22,8,8,2000,2024-01-15,2024-01-01,14,Cash,Late
23,8,8,2000,2024-02-18,2024-02-01,17,Cash,Late
24,8,8,2000,2024-03-20,2024-03-01,19,Cash,Late
25,9,9,2300,2024-01-01,2024-01-01,0,Bank Transfer,Paid
26,9,9,2300,2024-02-01,2024-02-01,0,Bank Transfer,Paid
27,9,9,2300,2024-03-01,2024-03-01,0,Bank Transfer,Paid
28,10,10,2400,2024-01-03,2024-01-01,2,Credit Card,Paid
29,10,10,2400,2024-02-05,2024-02-01,4,Credit Card,Paid
30,10,10,2400,2024-03-07,2024-03-01,6,Credit Card,Paid
31,11,11,2100,2024-01-01,2024-01-01,0,Bank Transfer,Paid
32,11,11,2100,2024-02-01,2024-02-01,0,Bank Transfer,Paid
33,11,11,2100,2024-03-01,2024-03-01,0,Bank Transfer,Paid
34,12,12,2150,2024-01-07,2024-01-01,6,Check,Paid
35,12,12,2150,2024-02-09,2024-02-01,8,Check,Paid
36,12,12,2150,2024-03-11,2024-03-01,10,Check,Late
37,13,13,1900,2024-01-01,2024-01-01,0,Bank Transfer,Paid
38,13,13,1900,2024-02-01,2024-02-01,0,Bank Transfer,Paid
39,13,13,1900,2024-03-01,2024-03-01,0,Bank Transfer,Paid
40,14,14,1950,2024-01-10,2024-01-01,9,Credit Card,Paid
41,14,14,1950,2024-02-12,2024-02-01,11,Credit Card,Paid
42,14,14,1950,2024-03-14,2024-03-01,13,Credit Card,Late
43,15,15,2400,2024-01-01,2024-01-01,0,Bank Transfer,Paid
44,15,15,2400,2024-02-01,2024-02-01,0,Bank Transfer,Paid
45,15,15,2400,2024-03-01,2024-03-01,0,Bank Transfer,Paid"""

# ============================================================
# MAINTENANCE
# ============================================================
maintenance_data = """ticket_id,tenant_id,property_id,issue_type,priority,complaint_date,resolved_date,days_to_resolve,cost,status,resolution_notes
1,1,1,Plumbing,High,2024-01-05,2024-01-20,15,500,Resolved,Pipe replaced
2,1,1,HVAC,Medium,2024-02-10,2024-02-22,12,300,Resolved,Filter cleaned
3,2,1,Electrical,High,2024-01-15,2024-01-24,9,400,Resolved,Wiring fixed
4,2,1,Plumbing,Low,2024-02-20,2024-02-27,7,250,Resolved,Drain unclogged
5,4,2,HVAC,Medium,2024-01-08,2024-01-12,4,150,Resolved,Thermostat replaced
6,6,3,Appliance,Low,2024-02-01,2024-02-19,18,800,Resolved,Dishwasher repaired
7,6,3,Structural,High,2024-03-05,2024-03-19,14,600,Resolved,Wall crack sealed
8,8,4,Plumbing,High,2024-01-20,2024-02-09,20,1000,Resolved,Pipe burst repaired
9,8,4,Electrical,Medium,2024-02-25,2024-03-12,15,750,Resolved,Circuit breaker replaced
10,10,5,HVAC,Low,2024-01-10,2024-01-13,3,200,Resolved,AC filter replaced
11,3,2,Appliance,Low,2024-02-15,2024-02-18,3,150,Resolved,Microwave replaced
12,5,3,Plumbing,Medium,2024-01-20,2024-01-25,5,300,Resolved,Faucet replaced
13,7,4,Structural,Low,2024-02-10,2024-02-13,3,100,Resolved,Door hinge fixed
14,9,5,HVAC,High,2024-03-01,2024-03-10,9,450,Resolved,AC unit replaced
15,11,6,Electrical,Medium,2024-01-25,2024-02-01,7,350,Resolved,Outlet replaced"""

# ============================================================
# CONVERT TO PANDAS
# ============================================================
properties_pd  = pd.read_csv(StringIO(properties_data))
tenants_pd     = pd.read_csv(StringIO(tenants_data))
leases_pd      = pd.read_csv(StringIO(leases_data))
payments_pd    = pd.read_csv(StringIO(payments_data))
maintenance_pd = pd.read_csv(StringIO(maintenance_data))

# ============================================================
# CONVERT TO SPARK
# ============================================================
properties  = spark.createDataFrame(properties_pd)
tenants     = spark.createDataFrame(tenants_pd)
leases      = spark.createDataFrame(leases_pd)
payments    = spark.createDataFrame(payments_pd)
maintenance = spark.createDataFrame(maintenance_pd)

# ============================================================
# VERIFY
# ============================================================
print("\nDATA LOADED SUCCESSFULLY")
print("="*50)
print(f"Properties  : {properties.count()} rows")
print(f"Tenants     : {tenants.count()} rows")
print(f"Leases      : {leases.count()} rows")
print(f"Payments    : {payments.count()} rows")
print(f"Maintenance : {maintenance.count()} rows")

print("\nColumns:")
print("Properties  :", properties.columns)
print("Tenants     :", tenants.columns)
print("Leases      :", leases.columns)
print("Payments    :", payments.columns)
print("Maintenance :", maintenance.columns)

CREATING ALL DATASETS

DATA LOADED SUCCESSFULLY
Properties  : 10 rows
Tenants     : 15 rows
Leases      : 15 rows
Payments    : 45 rows
Maintenance : 15 rows

Columns:
Properties  : ['property_id', 'property_name', 'city', 'state', 'total_units', 'monthly_rent', 'property_type', 'year_built']
Tenants     : ['tenant_id', 'tenant_name', 'email', 'phone', 'move_in_date', 'employment_status', 'credit_score']
Leases      : ['lease_id', 'tenant_id', 'property_id', 'unit_number', 'monthly_rent', 'original_rent', 'lease_start', 'lease_end', 'status', 'lease_type', 'renewed']
Payments    : ['payment_id', 'tenant_id', 'lease_id', 'amount', 'payment_date', 'due_date', 'days_late', 'payment_method', 'payment_status']
Maintenance : ['ticket_id', 'tenant_id', 'property_id', 'issue_type', 'priority', 'complaint_date', 'resolved_date', 'days_to_resolve', 'cost', 'status', 'resolution_notes']


## Data Cleaning & Transformation

In this step, we clean and standardize multiple datasets including payments, leases, maintenance, and properties.

### Key Transformations:
- Removed duplicate records using primary keys
- Filtered invalid or missing data
- Categorized payment behavior based on delay
- Identified partial payments
- Calculated rent increase percentages
- Computed lease expiry timelines
- Classified maintenance resolution efficiency

### Business Impact:
These transformations help in:
- Identifying late-paying tenants
- Tracking lease performance and rent changes
- Monitoring operational efficiency in maintenance
- Ensuring high-quality, reliable data for analytics

In [3]:
from pyspark.sql import functions as F

# Clean payments
payments_clean = payments \
    .dropDuplicates(['payment_id']) \
    .filter(F.col('amount') > 0) \
    .filter(F.col('tenant_id').isNotNull()) \
    .withColumn('payment_category',
        F.when(F.col('days_late') == 0,  'ON_TIME')
        .when(F.col('days_late') <= 5,   'GRACE')
        .when(F.col('days_late') <= 30,  'LATE')
        .otherwise('CRITICAL')
    ) \
    .withColumn('is_partial',
        F.when(F.col('amount') < 2000, 'YES')
        .otherwise('NO')
    )

# Clean leases
leases_clean = leases \
    .dropDuplicates(['lease_id']) \
    .filter(F.col('status') == 'active') \
    .withColumn('rent_increase_pct',
        F.round(
            (F.col('monthly_rent') -
             F.col('original_rent')) *
            100.0 / F.col('original_rent'),
            2
        )
    ) \
    .withColumn('days_to_expiry',
        F.datediff(
            F.to_date(F.col('lease_end')),
            F.current_date()
        )
    )

# Clean maintenance
maintenance_clean = maintenance \
    .dropDuplicates(['ticket_id']) \
    .filter(F.col('cost') > 0) \
    .withColumn('resolution_category',
        F.when(F.col('days_to_resolve') <= 3,  'Fast')
        .when(F.col('days_to_resolve') <= 7,   'Normal')
        .when(F.col('days_to_resolve') <= 14,  'Slow')
        .otherwise('Critical')
    )

# Clean properties
properties_clean = properties \
    .dropDuplicates(['property_id']) \
    .filter(F.col('total_units') > 0) \
    .filter(F.col('monthly_rent') > 0)

# Verify all cleaned
print("="*50)
print("CLEANING COMPLETE")
print("="*50)

print("\nPayment Category Breakdown:")
payments_clean \
    .groupBy('payment_category') \
    .count() \
    .orderBy('payment_category') \
    .show()

print("Lease Sample:")
leases_clean.select(
    'lease_id',
    'tenant_id',
    'monthly_rent',
    'rent_increase_pct',
    'days_to_expiry'
).show(5)

print("Maintenance Resolution:")
maintenance_clean \
    .groupBy('resolution_category') \
    .count() \
    .orderBy('resolution_category') \
    .show()

print("Properties:")
properties_clean.show(5)

CLEANING COMPLETE

Payment Category Breakdown:
+----------------+-----+
|payment_category|count|
+----------------+-----+
|           GRACE|    5|
|            LATE|   19|
|         ON_TIME|   21|
+----------------+-----+

Lease Sample:
+--------+---------+------------+-----------------+--------------+
|lease_id|tenant_id|monthly_rent|rent_increase_pct|days_to_expiry|
+--------+---------+------------+-----------------+--------------+
|       1|        1|        2200|             10.0|          -491|
|       2|        2|        2500|            13.64|          -675|
|       3|        3|        1800|              0.0|          -583|
|       4|        4|        2100|            10.53|          -491|
|       5|        5|        2500|              8.7|          -766|
+--------+---------+------------+-----------------+--------------+
only showing top 5 rows
Maintenance Resolution:
+-------------------+-----+
|resolution_category|count|
+-------------------+-----+
|           Critical|    4|


In [5]:
# Join payments with leases
payments_leases = payments_clean.join(
    leases_clean,
    "tenant_id",
    "left"
)

# Join with properties
final_df = payments_leases.join(
    properties_clean,
    "property_id",
    "left"
)

final_df.show()

tenant_risk = final_df.groupBy("tenant_id").agg(
    F.avg("days_late").alias("avg_delay")
).withColumn(
    "risk_level",
    F.when(F.col("avg_delay") > 10, "HIGH")
     .when(F.col("avg_delay") > 5, "MEDIUM")
     .otherwise("LOW")
)

tenant_risk.show()



+-----------+---------+----------+--------+------+------------+----------+---------+--------------+--------------+----------------+----------+--------+-----------+------------+-------------+-----------+----------+------+----------+-------+-----------------+--------------+-----------------+-------+-----+-----------+------------+-------------+----------+
|property_id|tenant_id|payment_id|lease_id|amount|payment_date|  due_date|days_late|payment_method|payment_status|payment_category|is_partial|lease_id|unit_number|monthly_rent|original_rent|lease_start| lease_end|status|lease_type|renewed|rent_increase_pct|days_to_expiry|    property_name|   city|state|total_units|monthly_rent|property_type|year_built|
+-----------+---------+----------+--------+------+------------+----------+---------+--------------+--------------+----------------+----------+--------+-----------+------------+-------------+-----------+----------+------+----------+-------+-----------------+--------------+-----------------+

In [7]:
# Create database first
spark.sql("CREATE DATABASE IF NOT EXISTS real_estate")

# Save as permanent tables
properties_clean.write \
    .mode("overwrite") \
    .saveAsTable("real_estate.properties")

payments_clean.write \
    .mode("overwrite") \
    .saveAsTable("real_estate.payments")

leases_clean.write \
    .mode("overwrite") \
    .saveAsTable("real_estate.leases")

maintenance_clean.write \
    .mode("overwrite") \
    .saveAsTable("real_estate.maintenance")

tenants.write \
    .mode("overwrite") \
    .saveAsTable("real_estate.tenants")

# Verify
spark.sql("SHOW TABLES IN real_estate").show()

+-----------+-----------+-----------+
|  namespace|  tableName|isTemporary|
+-----------+-----------+-----------+
|real_estate|     leases|      false|
|real_estate|maintenance|      false|
|real_estate|   payments|      false|
|real_estate| properties|      false|
|real_estate|    tenants|      false|
+-----------+-----------+-----------+



In [15]:
spark.sql("""
    SELECT * FROM real_estate.payments
    LIMIT 5
""").show()

spark.sql("""
-- ============================================
-- PROPERTY PERFORMANCE SUMMARY (CORRECT)
-- ============================================

SELECT l.property_id,
       SUM(p.amount) AS total_revenue,
       AVG(p.days_late) AS avg_delay,
       COUNT(p.payment_id) AS total_payments,
       SUM(CASE WHEN p.days_late > 0 THEN 1 ELSE 0 END) AS late_payments
FROM real_estate.payments p
JOIN real_estate.leases l
    ON p.tenant_id = l.tenant_id
GROUP BY l.property_id
ORDER BY total_revenue DESC;
""").show()

+----------+---------+--------+------+------------+----------+---------+--------------+--------------+----------------+----------+
|payment_id|tenant_id|lease_id|amount|payment_date|  due_date|days_late|payment_method|payment_status|payment_category|is_partial|
+----------+---------+--------+------+------------+----------+---------+--------------+--------------+----------------+----------+
|         1|        1|       1|  2200|  2024-01-03|2024-01-01|        2| Bank Transfer|          Paid|           GRACE|        NO|
|         2|        1|       1|  2200|  2024-02-06|2024-02-01|        5| Bank Transfer|          Paid|           GRACE|        NO|
|         3|        1|       1|  2200|  2024-03-09|2024-03-01|        8| Bank Transfer|          Paid|            LATE|        NO|
|         4|        2|       2|  2500|  2024-01-10|2024-01-01|        9|         Check|          Paid|            LATE|        NO|
|         5|        2|       2|  2500|  2024-02-12|2024-02-01|       11|         Ch

In [11]:
final_gold = final_df.groupBy("property_id").agg(
    F.sum("amount").alias("total_revenue"),
    F.avg("days_late").alias("avg_delay"),
    F.count("payment_id").alias("total_payments")
)

final_gold.write \
    .mode("overwrite") \
    .saveAsTable("real_estate.gold_property_summary")
spark.sql("""
    SELECT * FROM real_estate.gold_property_summary
    LIMIT 5
""").show()

+-----------+-------------+-----------------+--------------+
|property_id|total_revenue|        avg_delay|total_payments|
+-----------+-------------+-----------------+--------------+
|          7|        11550|              5.5|             6|
|          6|        12750|              4.0|             6|
|          5|        14100|              2.0|             6|
|          1|        14100|              7.0|             6|
|          3|        14100|6.833333333333333|             6|
+-----------+-------------+-----------------+--------------+



## End-to-End Architecture

1. Data Ingestion: Raw datasets created and loaded into PySpark
2. Data Processing: Cleaning and transformation using PySpark
3. Data Integration: Joining multiple datasets
4. Data Storage: Persisted as SQL tables in a structured database
5. Analytics Layer: SQL queries used for reporting and insights

This pipeline simulates a real-world data engineering workflow.